# BankSathi · Gemma-2B QLoRA on Colab T4

End-to-end runbook: dataset → pre-GPU audit → QLoRA fine-tune → 5-KPI hold-out gate → INT4 MediaPipe `.task`.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. Accept the licence at https://huggingface.co/google/gemma-2-2b-it and create a read token.
3. Have `gemma_training.zip` ready to upload (or put it in Google Drive).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1 · Upload code, install, log in (~2 min)

In [ ]:
import os
if not os.path.exists('/content/gemma_training'):
    from google.colab import files
    uploaded = files.upload()  # choose gemma_training.zip
    !unzip -o -q gemma_training.zip -d /content
%cd /content/gemma_training
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao  # Colab's preinstalled torchao is too old for peft and breaks the LoRA merge

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste HF token (after accepting google/gemma-2-2b-it terms)

## Step 2 · Generate 1,500-row stratified dataset + pre-GPU audit (~5 s)
Must show `valid_json_pct: 100.0`, `devanagari_parity_pct: 100.0`, `train_eval_leakage_count: 0` and `RESULT: PASS`.

In [ ]:
!python3 dataset_generator.py --out-dir data --target-rows 1500
!python3 eval_gemma_banksathi.py --train-file data/train.jsonl --eval-file data/eval.jsonl --report-file data/audit_report.json

In [ ]:
import json
rows = [json.loads(l) for l in open('data/train.jsonl', encoding='utf-8')]
for b in ['withdraw_bc_le_10k', 'safe_abstention', 'assistant_scam_alert', 'scam_safe']:
    r = next(x for x in rows if x['bucket'] == b)
    print(f"--- {b} [{r['register']}] ---\nQ: {r['query']}\nA: {r['completion']}\n")

## Step 3 · QLoRA fine-tune + merge to FP16 (~22-30 min on T4)

In [ ]:
!python3 train_gemma2b_qlora.py --model-id google/gemma-2-2b-it \
    --train-file data/train.jsonl --eval-file data/eval.jsonl \
    --epochs 4 --output-dir outputs/gemma2-2b-banksathi

## Step 4 · Live 5-KPI hold-out safety benchmark (~3 min)
Gates: JSON ≥ 99 · intent ≥ 96 · ScamGuard recall = 100 · abstention ≥ 95 · bilingual TTS ≥ 98.
If a gate fails, see the failure playbook in README.md, adjust, and re-run Steps 2-4.

In [ ]:
!python3 eval_gemma_banksathi.py \
    --model-dir outputs/gemma2-2b-banksathi/merged_fp16 \
    --eval-file data/eval.jsonl \
    --save-predictions outputs/eval_predictions.jsonl \
    --report-file outputs/kpi_report.json

## Save to Google Drive (Colab disks are wiped when the runtime ends)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/banksathi
!cp -r data outputs/gemma2-2b-banksathi/adapter outputs/gemma2-2b-banksathi/train_metrics.json outputs/kpi_report.json /content/drive/MyDrive/banksathi/
!cp -r outputs/gemma2-2b-banksathi/merged_fp16 /content/drive/MyDrive/banksathi/
!ls -la /content/drive/MyDrive/banksathi

## Step 5 · INT4 MediaPipe `.task` conversion
`ai-edge-torch` pins its own torch build, so **do `Runtime → Restart session`** (or `Disconnect and delete runtime` for a clean one; a High-RAM CPU runtime is enough) and run only the cells below.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/gemma_training'):
    from google.colab import files
    files.upload()  # gemma_training.zip again if this is a new runtime
    !unzip -o -q gemma_training.zip -d /content
%cd /content/gemma_training
!pip install -q -r requirements-convert.txt

In [ ]:
!python3 convert_to_mediapipe.py \
    --merged-dir /content/drive/MyDrive/banksathi/merged_fp16 \
    --context-len 512 \
    --output-task /content/drive/MyDrive/banksathi/gemma2-2b-banksathi-cpu-int4.task

Download `gemma2-2b-banksathi-cpu-int4.task` (~1.45 GB) from Drive, then load it in BankSathi:

* **In-app:** Settings → *Import Gemma-2B (.task / .bin)*
* **ADB:** `adb push gemma2-2b-banksathi-cpu-int4.task /data/local/tmp/llm/gemma2-2b-it-cpu-int4.task`

Then run the 6-check airplane-mode smoke test in README.md.